# BIRD Graph Ablation — Retrieval (N=8) + Minimal Connecting Subgraph

**Component 2** of the deterministic schema-linking ablation study. Layered on top of the [retrieval ablation](retrieval.md). Same SQL generator (Qwen2.5-Coder-32B-Instruct, 4-bit NF4), same expansion model (Qwen2.5-7B-Instruct), same BIRD dev set. Two changes:

1. **Retrieval N bumped 5 → 8.** Retrieval becomes recall-favoring (~97-99% on every BIRD dev DB).
2. **Graph step inserted** between retrieval and generation. Computes the *minimal connecting subgraph* over the 8 candidates using BIRD's foreign-key structure: preserves anchor tables, adds *bridge tables* on FK paths between them, separates *unresolved* candidates.

Pipeline:
```
question + evidence
      ↓
[Qwen-7B] expand → 3 paraphrases + entity list
      ↓
[BM25 + dense + RRF] each variant → ranked tables (uncapped)
      ↓
RRF aggregate across variants → top-N=8 candidate tables
      ↓
[SchemaGraph.minimal_connecting_subgraph] →
    core tables + bridge tables + approved-joins list
      ↓
build linked-schema prompt:
    - flat table list (graph-natural order, not RRF rank)
    - explicit approved-joins block (FK predicates the model MUST use)
      ↓
[Qwen-32B] → SQL
```

**Comparison frame:**

| Method | Join Acc. | Exec. Acc. |
|---|---|---|
| Baseline (Qwen-32B, full schema) | 66.3 | 54.2 |
| Retrieval (N=5) | 66.7 | 52.1 |
| **+ Graph (this run, N=8 → graph)** | **?** | **?** |

**Hypothesis.** Graph fixes the precision losses identified in `retrieval.md` §3:
- **Source 2 (over-join pressure):** approved-joins list constrains the model — can't fabricate joins outside the FK paths.
- **Source 3 (rank-demotion):** graph presents tables in flat structural order, not RRF rank.
- **Source 1 (recall failure):** partially mitigated by bridge-table insertion when missing tables are bridges; not when they are core entities. N=8 retrieval also reduces this floor vs N=5.

Predictions land in `/content/predict_dev_graph.json`. Per-question graph stats (anchors, bridges, joins, unresolved) logged to `/content/graph_log.jsonl` for diagnosis.

## 1. Setup — clone repo, install deps

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/mayursk2000/Stop-Guessing-Joins-Deterministic-Schema-Linking-for-Text-to-SQL-Agents.git"
REPO_DIR = pathlib.Path("/content/Stop-Guessing-Joins-Deterministic-Schema-Linking-for-Text-to-SQL-Agents")

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "sqlglot", "huggingface_hub", "requests", "tqdm", "psutil",
    "transformers>=4.45", "accelerate>=0.34", "bitsandbytes>=0.43", "sentencepiece",
    "rank_bm25", "sentence-transformers",
])
print("Repo:", REPO_DIR)

## 2. Secrets — `HF_TOKEN` from Colab Secrets

In [ ]:
HF_TOKEN = None
try:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as e:
        print("HF_TOKEN not available:", e)
except ImportError:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print("HF_TOKEN set:", bool(HF_TOKEN))

## 3. GPU check

In [ ]:
import torch

GEN_MODEL_NAME       = "unsloth/Qwen2.5-Coder-32B-Instruct-bnb-4bit"  # pre-4bit, ~18 GB disk
EXPANSION_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"                     # BF16, ~14 GB
ENCODER_NAME         = "BAAI/bge-small-en-v1.5"

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name}  |  VRAM: {vram_gb:.1f} GB")
else:
    print("No GPU detected.")

print("Generator:", GEN_MODEL_NAME)
print("Expansion:", EXPANSION_MODEL_NAME)
print("Encoder:  ", ENCODER_NAME)

## 4. Download BIRD dev — same downloader as retrieval

In [ ]:
import re, zipfile, json, urllib.request

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
import gdown

BIRD_ROOT = pathlib.Path("/content/bird")
BIRD_ROOT.mkdir(exist_ok=True)

BIRD_DEV_URL = "https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip"
zip_path = BIRD_ROOT / "dev.zip"

def _looks_like_html(p: pathlib.Path) -> bool:
    if not p.exists() or p.stat().st_size < 1_000_000:
        try:
            head = p.read_bytes()[:512].lower()
            return b"<!doctype html" in head or b"<html" in head
        except Exception:
            return True
    return False

def _drive_id(url: str):
    for pat in (r"/file/d/([A-Za-z0-9_-]{20,})", r"[?&]id=([A-Za-z0-9_-]{20,})"):
        m = re.search(pat, url)
        if m:
            return m.group(1)
    return None

if zip_path.exists() and _looks_like_html(zip_path):
    zip_path.unlink()

if BIRD_DEV_URL and not zip_path.exists():
    drive_id = _drive_id(BIRD_DEV_URL)
    if drive_id:
        gdown.download(id=drive_id, output=str(zip_path), quiet=False)
    else:
        urllib.request.urlretrieve(BIRD_DEV_URL, zip_path)

assert zip_path.exists() and not _looks_like_html(zip_path), "BIRD dev download failed."

if not list(BIRD_ROOT.rglob("dev.json")):
    print("Extracting ...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(BIRD_ROOT)
    for inner in BIRD_ROOT.rglob("dev_databases.zip"):
        with zipfile.ZipFile(inner) as z:
            z.extractall(inner.parent)
        inner.unlink()
    zip_path.unlink()
    print("Removed BIRD zips after extraction (~3 GB freed).")

candidates = [p.parent for p in BIRD_ROOT.rglob("dev.json")]
assert candidates
BIRD_DEV = candidates[0]
BIRD_DBS = next(BIRD_DEV.rglob("dev_databases"), BIRD_DEV / "dev_databases")
print("BIRD_DEV =", BIRD_DEV)
print("BIRD_DBS =", BIRD_DBS)
print("Questions:", len(json.loads((BIRD_DEV / "dev.json").read_text())))
print("Databases:", len(list(BIRD_DBS.iterdir())) if BIRD_DBS.exists() else "missing")

## 5. Schema introspection (same as retrieval)

In [ ]:
import sqlite3, csv
from text2sql_agent_prototype.prototype import Schema, Table, Column, ForeignKey

_BIRD_TABLES_JSON = None

def _load_dev_tables() -> dict:
    global _BIRD_TABLES_JSON
    if _BIRD_TABLES_JSON is None:
        path = next(BIRD_DEV.rglob("dev_tables.json"), None)
        _BIRD_TABLES_JSON = {b["db_id"]: b for b in json.loads(path.read_text())} if path else {}
    return _BIRD_TABLES_JSON

def _read_column_descriptions(db_id: str):
    desc_dir = BIRD_DBS / db_id / "database_description"
    out = {}
    if not desc_dir.exists():
        return out
    for csv_path in desc_dir.glob("*.csv"):
        tbl = csv_path.stem.lower()
        col_descs = {}
        try:
            with open(csv_path, encoding="utf-8", errors="replace") as f:
                for row in csv.DictReader(f):
                    col = (row.get("original_column_name") or "").strip()
                    desc = (row.get("column_description") or "").strip()
                    val_desc = (row.get("value_description") or "").strip()
                    full = " ; ".join(filter(None, [desc, val_desc]))
                    if col:
                        col_descs[col.lower()] = full
        except Exception as exc:
            print(f"  warn: {csv_path.name}: {exc}")
        out[tbl] = col_descs
    return out

_SCHEMA_CACHE = {}
def schema_from_bird(db_id: str) -> Schema:
    if db_id in _SCHEMA_CACHE:
        return _SCHEMA_CACHE[db_id]
    db_path = BIRD_DBS / db_id / f"{db_id}.sqlite"
    conn = sqlite3.connect(str(db_path))
    try:
        cur = conn.cursor()
        table_names = [r[0] for r in cur.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
        ).fetchall()]
        col_descs = _read_column_descriptions(db_id)
        bird_meta = _load_dev_tables().get(db_id)

        tables = {}
        for tname in table_names:
            cols_info = cur.execute(f'PRAGMA table_info("{tname}")').fetchall()
            tbl_descs = col_descs.get(tname.lower(), {})
            columns = tuple(
                Column(name=c[1],
                       description=" | ".join(filter(None, [
                           c[2].lower() if c[2] else "",
                           tbl_descs.get(c[1].lower(), ""),
                       ])))
                for c in cols_info
            )
            tbl_desc = tname.replace("_", " ")
            if bird_meta and tname in bird_meta.get("table_names_original", []):
                idx = bird_meta["table_names_original"].index(tname)
                if idx < len(bird_meta.get("table_names", [])):
                    tbl_desc = bird_meta["table_names"][idx] or tbl_desc
            tables[tname] = Table(name=tname, columns=columns, description=tbl_desc)

        foreign_keys = []
        if bird_meta:
            names_orig = bird_meta["table_names_original"]
            cols_orig  = bird_meta["column_names_original"]
            for fk_pair in bird_meta.get("foreign_keys", []):
                l_idx, r_idx = fk_pair[0], fk_pair[1]
                lt = names_orig[cols_orig[l_idx][0]]; lc = cols_orig[l_idx][1]
                rt = names_orig[cols_orig[r_idx][0]]; rc = cols_orig[r_idx][1]
                if lt in tables and rt in tables:
                    foreign_keys.append(ForeignKey(left_table=lt, left_column=lc,
                                                   right_table=rt, right_column=rc))
        seen = {(fk.left_table, fk.left_column, fk.right_table, fk.right_column) for fk in foreign_keys}
        for tname in table_names:
            for fk in cur.execute(f'PRAGMA foreign_key_list("{tname}")').fetchall():
                _, _, ref_table, from_col, to_col, *_ = fk
                if ref_table in tables and (tname, from_col, ref_table, to_col) not in seen:
                    foreign_keys.append(ForeignKey(left_table=tname, left_column=from_col,
                                                   right_table=ref_table, right_column=to_col))

        sch = Schema(tables=tables, foreign_keys=foreign_keys)
        _SCHEMA_CACHE[db_id] = sch
        return sch
    finally:
        conn.close()

def load_dev(bird_dev_dir: pathlib.Path) -> list:
    return json.loads((bird_dev_dir / "dev.json").read_text())

## 6. Sanity check

In [ ]:
print("BIRD_DEV =", BIRD_DEV)
questions = load_dev(BIRD_DEV)
print(f"Total questions: {len(questions)}")
sch0 = schema_from_bird(questions[0]["db_id"])
print(f"Sample db={questions[0]['db_id']}: {len(sch0.tables)} tables, {len(sch0.foreign_keys)} FKs")

## 7. Hybrid retriever (same code as retrieval.ipynb)

In [ ]:
import re as _re
from collections import defaultdict
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from text2sql_agent_prototype.prototype import (
    HybridRetriever, RetrievalResult, RetrievalMatch,
)

print(f"Loading encoder {ENCODER_NAME} ...")
encoder = SentenceTransformer(ENCODER_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
print("Encoder loaded.")

def _smart_split(text: str):
    out = []
    for word in _re.findall(r"[A-Za-z0-9]+", text):
        for piece in _re.findall(r"[A-Z]+(?=[A-Z][a-z])|[A-Z]?[a-z]+|[A-Z]+|[0-9]+", word):
            t = piece.lower()
            if len(t) >= 2:
                out.append(t)
    return out


class BIRDHybridRetriever(HybridRetriever):
    RRF_K = 60

    def __init__(self, schema: Schema, db_id: str, top_k: int = 12):
        self.schema = schema
        self.db_id = db_id
        self.top_k = top_k
        self.min_score = 0.0

        self.docs = []
        self.doc_tables = []
        for tname, table in schema.tables.items():
            for c in table.columns:
                self.docs.append(f"Table {tname} ({table.description}). Column {c.name}: {c.description or ''}")
                self.doc_tables.append(tname)

        self.bm25 = BM25Okapi([_smart_split(d) for d in self.docs])
        self.dense = encoder.encode(self.docs, normalize_embeddings=True,
                                    show_progress_bar=False, convert_to_numpy=True)

    def _rank_tables(self, query: str):
        if not self.docs:
            return []
        bm25_scores = self.bm25.get_scores(_smart_split(query))
        bm25_rank = {int(i): r for r, i in enumerate(np.argsort(-bm25_scores))}
        q_emb = encoder.encode([query], normalize_embeddings=True,
                               show_progress_bar=False, convert_to_numpy=True)[0]
        dense_scores = self.dense @ q_emb
        dense_rank = {int(i): r for r, i in enumerate(np.argsort(-dense_scores))}
        K = self.RRF_K
        rrf = {i: 1.0/(K+bm25_rank[i]) + 1.0/(K+dense_rank[i]) for i in range(len(self.docs))}
        table_score = {}
        for i, tname in enumerate(self.doc_tables):
            s = rrf[i]
            if tname not in table_score or s > table_score[tname]:
                table_score[tname] = s
        return sorted(table_score.items(), key=lambda kv: -kv[1])

    def retrieve(self, query: str) -> RetrievalResult:
        ranked = self._rank_tables(query)[:self.top_k]
        return RetrievalResult(
            query=query,
            candidate_tables=[t for t, _ in ranked],
            matches=[RetrievalMatch(table=t, lexical_score=0.0, semantic_score=0.0,
                                    score=float(s), matched_terms=[]) for t, s in ranked],
        )


_RETRIEVER_CACHE = {}
def get_retriever(db_id: str) -> BIRDHybridRetriever:
    if db_id not in _RETRIEVER_CACHE:
        _RETRIEVER_CACHE[db_id] = BIRDHybridRetriever(schema_from_bird(db_id), db_id)
    return _RETRIEVER_CACHE[db_id]

## 8. Load Qwen-7B expansion model (same as retrieval)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import re, gc, json as _json

print(f"Loading expansion model {EXPANSION_MODEL_NAME} (BF16) ...")
exp_tokenizer = AutoTokenizer.from_pretrained(EXPANSION_MODEL_NAME, token=HF_TOKEN, trust_remote_code=True)
exp_tokenizer.padding_side = "left"
exp_tokenizer.truncation_side = "left"
if exp_tokenizer.pad_token_id is None:
    exp_tokenizer.pad_token = exp_tokenizer.eos_token
    exp_tokenizer.pad_token_id = exp_tokenizer.eos_token_id

exp_model = AutoModelForCausalLM.from_pretrained(
    EXPANSION_MODEL_NAME, token=HF_TOKEN,
    torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True,
)
exp_model.eval()
print("Expansion model loaded.")

EXPANSION_SYSTEM = (
    "You are a query-expansion assistant for a Text-to-SQL retrieval system. "
    "You receive a natural-language database question and optional evidence "
    "hints. Your job: emit JSON with two fields:\n"
    '  "paraphrases": 3 alternate phrasings of the question, each preserving '
    "intent but using different vocabulary.\n"
    '  "entities": short noun phrases pulled from the question/evidence that '
    "look like database tables, columns, or filter values. Include backticked "
    "tokens from the evidence verbatim (without backticks). 1-8 entities.\n\n"
    "Output ONLY the JSON object. No prose, no markdown fences."
)
EXPANSION_FEWSHOT = [
    {"role": "user", "content":
        "Question: How many charter schools in Fresno County have free meal counts above 100?\n"
        "Evidence: Charter schools refers to `Charter School (Y/N)` = 1 in the frpm"},
    {"role": "assistant", "content":
        '{"paraphrases": ["Count charter schools located in Fresno County with '
        'free meal counts greater than 100", "How many institutions categorized '
        'as charter in Fresno County have more than 100 free meals", "Number '
        'of charter-status schools in the Fresno County area where free meal '
        'count exceeds 100"], "entities": ["charter schools", "Fresno County", '
        '"free meal count", "Charter School (Y/N)", "frpm"]}'},
]

_JSON_RE = re.compile(r"\{.*\}", re.DOTALL)

def _parse_expansion(raw: str, fallback_question: str):
    m = _JSON_RE.search(raw)
    if not m:
        return {"paraphrases": [fallback_question], "entities": []}
    try:
        obj = _json.loads(m.group(0))
    except Exception:
        return {"paraphrases": [fallback_question], "entities": []}
    paras = obj.get("paraphrases") or []
    if not isinstance(paras, list):
        paras = [str(paras)]
    paras = [str(p).strip() for p in paras if str(p).strip()][:3] or [fallback_question]
    ents = obj.get("entities") or []
    if not isinstance(ents, list):
        ents = [str(ents)]
    ents = [str(e).strip() for e in ents if str(e).strip()][:8]
    return {"paraphrases": paras, "entities": ents}

def expand_query_batch(items, batch_size: int = 16):
    if not items:
        return []
    results = []
    for start in range(0, len(items), batch_size):
        chunk = items[start:start + batch_size]
        prompts = []
        for q, ev in chunk:
            user = f"Question: {q}" + (f"\nEvidence: {ev}" if ev else "")
            msgs = [
                {"role": "system", "content": EXPANSION_SYSTEM},
                *EXPANSION_FEWSHOT,
                {"role": "user", "content": user},
            ]
            prompts.append(exp_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
        try:
            inputs = exp_tokenizer(prompts, return_tensors="pt", padding=True,
                                    truncation=True, max_length=2048).to(exp_model.device)
            with torch.inference_mode():
                out = exp_model.generate(**inputs, max_new_tokens=256, do_sample=False,
                                         pad_token_id=exp_tokenizer.pad_token_id)
            decoded = exp_tokenizer.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
            del out, inputs
            try: torch.cuda.empty_cache()
            except Exception: pass
        except BaseException as exc:
            if isinstance(exc, (KeyboardInterrupt, SystemExit)):
                raise
            print(f"  expansion batch failed ({type(exc).__name__}: {str(exc)[:80]})")
            decoded = [""] * len(chunk)
        for raw, (q, _ev) in zip(decoded, chunk):
            results.append(_parse_expansion(raw, q))
    return results

print("expand_query_batch ready.")

## 9. Multi-query retrieval at N=8 + minimal connecting subgraph

Two-step schema linking:

1. **Retrieval (N=8)** — recall-favoring superset via RRF over expansion variants.
2. **Graph step** — `SchemaGraph.minimal_connecting_subgraph(candidates)` returns a `JoinPlan` with:
   - `tables`: connected core tables in graph-natural order (BFS from first anchor) + bridge tables on FK paths
   - `joins`: list of `ForeignKey` predicates the model is allowed to use
   - `unresolved_tables`: candidates the graph couldn't connect (kept as fallback context)

The graph removes RRF rank order — tables are presented flat. Bridge tables retrieval missed get added via FK traversal.

In [ ]:
from text2sql_agent_prototype.prototype import SchemaGraph, JoinPlan

N_RETRIEVAL_TABLES = 8  # bumped from 5 for recall headroom; graph prunes structurally

_GRAPH_CACHE = {}
def get_graph(db_id: str) -> SchemaGraph:
    if db_id not in _GRAPH_CACHE:
        _GRAPH_CACHE[db_id] = SchemaGraph(schema_from_bird(db_id))
    return _GRAPH_CACHE[db_id]


def retrieve_then_graph(db_id: str, question: str, evidence: str, expansion: dict,
                       n_retrieval: int = N_RETRIEVAL_TABLES):
    """Run retrieval (RRF over expansion variants) → graph (minimal connecting subgraph).

    Returns (join_plan, debug_dict) where:
      - join_plan.tables: graph-resolved tables (anchors + bridges, flat order)
      - join_plan.joins: approved FK joins among them
      - join_plan.unresolved_tables: candidates graph couldn't connect
      - debug.retrieved: raw retrieval candidates (top-N_RETRIEVAL_TABLES)
      - debug.bridges: tables added by the graph that weren't in retrieved set
    """
    retriever = get_retriever(db_id)

    variants = []
    anchor = (question + " " + evidence).strip() if evidence else question
    variants.append(anchor)
    variants.extend(expansion.get("paraphrases", []))
    variants.extend(expansion.get("entities", []))

    K = 60
    table_rrf = defaultdict(float)
    for v in variants:
        if not v.strip():
            continue
        for rank, (tname, _s) in enumerate(retriever._rank_tables(v)):
            table_rrf[tname] += 1.0 / (K + rank)

    retrieved = [t for t, _ in sorted(table_rrf.items(), key=lambda kv: -kv[1])[:n_retrieval]]

    graph = get_graph(db_id)
    join_plan = graph.minimal_connecting_subgraph(retrieved)

    bridges = [t for t in join_plan.tables if t not in retrieved]

    return join_plan, {
        "n_variants": len(variants),
        "retrieved": retrieved,
        "graph_tables": list(join_plan.tables),
        "bridges": bridges,
        "unresolved": list(join_plan.unresolved_tables),
        "approved_joins": [j.join_condition() for j in join_plan.joins],
    }


# ---- Probe: graph on Q0 ----
q0 = questions[0]
probe_exp = {"paraphrases": [q0["question"]], "entities": []}
probe_plan, probe_dbg = retrieve_then_graph(
    q0["db_id"], q0["question"], q0.get("evidence", "") or "", probe_exp,
)
print(f"db={q0['db_id']}")
print(f"retrieved (top-{N_RETRIEVAL_TABLES}): {probe_dbg['retrieved']}")
print(f"graph tables:                       {probe_dbg['graph_tables']}")
print(f"bridges added by graph:             {probe_dbg['bridges'] or '(none)'}")
print(f"unresolved candidates:              {probe_dbg['unresolved'] or '(none)'}")
print(f"approved joins:                     {probe_dbg['approved_joins'] or '(no joins required)'}")

## 10. Load Qwen-32B and define graph-aware generator

The prompt now has **two structural sections**:

1. **Linked schema** — graph-resolved tables (anchors + bridges) with their full column lists. Flat order, no rank.
2. **Approved joins** — the explicit list of FK predicates the model is *required* to use when joining. Anything else is forbidden.

Plus the existing BIRD evidence + question. The system prompt is updated to enforce the approved-joins rule.

In [ ]:
from transformers import BitsAndBytesConfig
import re, gc

USE_4BIT          = True
MAX_NEW_TOKENS    = 256
MAX_INPUT_TOKENS  = 8192
MAX_DESC_LEN      = 100
MAX_TABLE_COLUMNS = 80

print(f"Loading {GEN_MODEL_NAME} (4-bit={USE_4BIT}) ...")
_bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
) if USE_4BIT else None

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME, token=HF_TOKEN, trust_remote_code=True)
gen_tokenizer.padding_side = "left"
gen_tokenizer.truncation_side = "left"
if gen_tokenizer.pad_token_id is None:
    gen_tokenizer.pad_token = gen_tokenizer.eos_token
    gen_tokenizer.pad_token_id = gen_tokenizer.eos_token_id

gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME, token=HF_TOKEN,
    quantization_config=_bnb_cfg, torch_dtype=torch.bfloat16,
    device_map="auto", trust_remote_code=True,
)
gen_model.eval()
print("Generator loaded.")

_FENCE = re.compile(r"^```(?:sql|sqlite)?\s*|\s*```$", re.IGNORECASE | re.MULTILINE)

GEN_SYSTEM_PROMPT = (
    "You are an expert Text-to-SQL assistant for SQLite, evaluated on BIRD. "
    "You receive: (1) a SCHEMA-LINKED set of tables (resolved by retrieval + "
    "FK graph traversal), (2) an APPROVED-JOINS list (the only legal join "
    "predicates), (3) optional BIRD evidence with column hints, (4) a natural-"
    "language question.\n\n"
    "STRICT RULES:\n"
    "- If the evidence names a specific column in backticks (e.g. "
    "`Free Meal Count (K-12)`), USE THAT EXACT COLUMN. Do not substitute a "
    "similar-sounding column even if it seems to mean the same thing.\n"
    "- If the evidence provides a formula (e.g. \"rate = A / B\"), COMPUTE "
    "that expression literally. When the formula is a ratio, CAST the numerator "
    "to REAL (e.g. CAST(A AS REAL) / B) so SQLite doesn't integer-divide.\n"
    "- If the evidence gives a literal value (e.g. \"= 1\" or \"= 'Directly funded'\"), "
    "use that EXACT value, including punctuation, capitalization, and spacing.\n"
    "- Use ONLY tables and columns that appear in the linked schema below.\n"
    "- When joining, you MUST use only predicates from the APPROVED-JOINS list. "
    "Do not fabricate join predicates outside that list.\n"
    "- The question may need only ONE table. Do not force a join unless required.\n"
    "- Quote column names containing spaces or special characters with double quotes.\n"
    "- Return EXACTLY ONE SQLite SELECT statement. No prose, no code fences."
)

def _trim(s: str) -> str:
    s = (s or "").replace("\r", " ").replace("\n", " ").replace("\t", " ").strip()
    if len(s) > MAX_DESC_LEN:
        s = s[:MAX_DESC_LEN].rstrip() + "…"
    return s

def linked_schema_block(db_id: str, join_plan: JoinPlan) -> str:
    """CREATE-TABLE-style block for graph-resolved tables, in graph order."""
    schema = schema_from_bird(db_id)
    blocks = []
    all_tables = list(dict.fromkeys([*join_plan.tables, *join_plan.unresolved_tables]))
    for tname in all_tables:
        if tname not in schema.tables:
            continue
        table = schema.tables[tname]
        lines = []
        for c in table.columns[:MAX_TABLE_COLUMNS]:
            desc = _trim(c.description or "")
            base = f'    "{c.name}"'
            lines.append(f"{base}  -- {desc}" if desc else base)
        if len(table.columns) > MAX_TABLE_COLUMNS:
            lines.append(f"    -- ... {len(table.columns) - MAX_TABLE_COLUMNS} more columns omitted")
        label = "" if tname in join_plan.tables else "  -- (unresolved candidate, no FK path to other tables)"
        blocks.append(f'CREATE TABLE "{tname}" ({label}\n' + ",\n".join(lines) + "\n);")
    return "\n\n".join(blocks)

def approved_joins_block(join_plan: JoinPlan) -> str:
    if not join_plan.joins:
        return "(no joins required — single-table query is acceptable if the answer needs only one table)"
    return "\n".join(f"  {j.join_condition()}" for j in join_plan.joins)

def _build_gen_messages(db_id: str, question: str, evidence: str, join_plan: JoinPlan):
    user_parts = [
        f"Linked schema (graph-resolved from retrieved candidates):\n{linked_schema_block(db_id, join_plan)}",
        f"Approved joins (use ONLY these to connect tables):\n{approved_joins_block(join_plan)}",
    ]
    if evidence:
        user_parts.append(
            "BIRD evidence — REQUIRED column hints. If a column or value here is "
            f"given in backticks, use it EXACTLY as written:\n{evidence}"
        )
    user_parts.append(f"Question:\n{question}")
    user_parts.append("Output only the SELECT statement.")
    return [
        {"role": "system", "content": GEN_SYSTEM_PROMPT},
        {"role": "user",   "content": "\n\n".join(user_parts)},
    ]

def _clean_sql(text: str) -> str:
    text = _FENCE.sub("", text).strip()
    m = re.search(r"\b(SELECT|WITH)\b", text, re.IGNORECASE)
    if m:
        text = text[m.start():]
    return text.strip().rstrip(";").strip()

def _validate_sql(sql: str) -> str:
    if not re.match(r"(?i)^\s*(select|with)\b", sql):
        raise ValueError(f"Non-SELECT output: {sql[:120]}")
    return sql

def _free_cache():
    try: torch.cuda.empty_cache()
    except Exception: pass

def generate_sql_one(db_id: str, question: str, evidence: str, join_plan: JoinPlan) -> str:
    msgs = _build_gen_messages(db_id, question, evidence, join_plan)
    prompt = gen_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS).to(gen_model.device)
    with torch.inference_mode():
        out = gen_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=gen_tokenizer.pad_token_id)
    raw = gen_tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    del out, inputs
    _free_cache()
    return _validate_sql(_clean_sql(raw))

def generate_sql_batch(items, max_input_length: int = MAX_INPUT_TOKENS):
    """items: list of (db_id, question, evidence, join_plan)."""
    if not items:
        return []
    out = inputs = new_token_rows = None
    try:
        prompts = [
            gen_tokenizer.apply_chat_template(
                _build_gen_messages(db, q, ev, jp),
                tokenize=False, add_generation_prompt=True,
            ) for (db, q, ev, jp) in items
        ]
        prompt_lengths = [len(gen_tokenizer(p, add_special_tokens=False).input_ids) for p in prompts]
        if prompt_lengths and max(prompt_lengths) > max_input_length:
            print(f"  truncating gen batch: max_tokens={max(prompt_lengths)} cap={max_input_length}")
        inputs = gen_tokenizer(prompts, return_tensors="pt", padding=True,
                               truncation=True, max_length=max_input_length).to(gen_model.device)
        with torch.inference_mode():
            out = gen_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                     pad_token_id=gen_tokenizer.pad_token_id)
        new_token_rows = out[:, inputs.input_ids.shape[1]:]
        decoded = gen_tokenizer.batch_decode(new_token_rows, skip_special_tokens=True)
    except BaseException as exc:
        if isinstance(exc, (KeyboardInterrupt, SystemExit)):
            raise
        print(f"  gen batch failed ({type(exc).__name__}: {str(exc)[:120]})  batch={len(items)}")
        del out, inputs
        _free_cache(); gc.collect()
        if len(items) > 1:
            mid = len(items) // 2
            return generate_sql_batch(items[:mid], max_input_length) + generate_sql_batch(items[mid:], max_input_length)
        try:
            return [generate_sql_one(*items[0])]
        except Exception:
            return ["SELECT 1"]
    finally:
        try: del out, inputs, new_token_rows
        except NameError: pass
        _free_cache()
    return [
        (lambda raw: _validate_sql(_clean_sql(raw)) if re.search(r"\b(SELECT|WITH)\b", raw, re.IGNORECASE) else "SELECT 1")(r)
        if r else "SELECT 1" for r in decoded
    ]

print("SQL generator ready.")

## 11. Smoke test — full retrieval + graph + generation on Q0

In [ ]:
from text2sql_agent_prototype.prototype import SchemaGraph

probe_exp = expand_query_batch([(q0["question"], q0.get("evidence", "") or "")])[0]
probe_plan, probe_dbg = retrieve_then_graph(
    q0["db_id"], q0["question"], q0.get("evidence", "") or "", probe_exp,
)
sql = generate_sql_one(q0["db_id"], q0["question"], q0.get("evidence", "") or "", probe_plan)
print(f"db_id:    {q0['db_id']}")
print(f"question: {q0['question']}")
print(f"evidence: {q0.get('evidence', '')}")
print(f"\nretrieved (top-{N_RETRIEVAL_TABLES}): {probe_dbg['retrieved']}")
print(f"graph tables:                       {probe_dbg['graph_tables']}")
print(f"bridges added:                      {probe_dbg['bridges']}")
print(f"approved joins:                     {probe_dbg['approved_joins']}")
print(f"\nGenerated SQL: {sql}")
print(f"Gold SQL:      {q0['SQL']}")

## 12. Run on full BIRD dev → `predict_dev_graph.json`

Per-question: expand → retrieve(N=8) → graph → generate. Predictions checkpointed every 50; per-question graph stats logged for failure diagnosis.

In [ ]:
from tqdm.auto import tqdm
import gc, psutil, os, json

N             = len(questions)
GEN_BATCH     = int(os.environ.get("GRAPH_GEN_BATCH", "8"))
EXP_BATCH     = int(os.environ.get("GRAPH_EXP_BATCH", "16"))
PRED_PATH     = pathlib.Path("/content/predict_dev_graph.json")
LOG_PATH      = pathlib.Path("/content/graph_log.jsonl")

predictions = {}
if PRED_PATH.exists():
    predictions = json.loads(PRED_PATH.read_text())
    print(f"Resuming with {len(predictions)} predictions")
if not predictions and LOG_PATH.exists():
    LOG_PATH.unlink()

def _mem_str() -> str:
    rss = psutil.Process(os.getpid()).memory_info().rss / 1e9
    avail = psutil.virtual_memory().available / 1e9
    gpu = ""
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        gpu = f"  GPU free={free/1e9:.1f}GB"
    return f"RSS={rss:.1f}GB  free={avail:.1f}GB{gpu}"

def write_preds(p, d):
    with open(p, "w", encoding="utf-8") as f:
        json.dump(d, f)

def append_log(p, lines):
    if not lines:
        return
    with open(p, "a", encoding="utf-8") as f:
        for r in lines:
            f.write(json.dumps(r) + "\n")

groups = defaultdict(list)
for i, q in enumerate(questions[:N]):
    groups[q["db_id"]].append((i, q))

print(f"BIRD dev: {N} questions across {len(groups)} dbs")
print(f"Memory at start: {_mem_str()}")

total_done = len(predictions)
pbar = tqdm(total=N, initial=total_done, desc="BIRD dev (graph)")
last_checkpoint = total_done

for db_id, qlist in groups.items():
    qlist_remaining = [(i, q) for (i, q) in qlist if str(q.get("question_id", i)) not in predictions]
    if not qlist_remaining:
        continue

    retr = get_retriever(db_id)
    graph = get_graph(db_id)
    print(f"  [{db_id}] {len(qlist_remaining)} questions, "
          f"corpus={len(retr.docs)} cols, fk_graph={sum(len(v) for v in graph.adjacent.values())//2} edges  {_mem_str()}")

    # Pass 1: expansion
    exp_inputs = [(q["question"], q.get("evidence", "") or "") for (_i, q) in qlist_remaining]
    expansions = expand_query_batch(exp_inputs, batch_size=EXP_BATCH)

    # Pass 2: retrieve + graph per question
    join_plans = []
    log_buf = []
    for (orig_i, q), exp in zip(qlist_remaining, expansions):
        qid = str(q.get("question_id", orig_i))
        plan, dbg = retrieve_then_graph(db_id, q["question"], q.get("evidence", "") or "", exp)
        join_plans.append(plan)
        log_buf.append({
            "qid": qid, "db_id": db_id,
            "paraphrases": exp.get("paraphrases", []),
            "entities":    exp.get("entities", []),
            **dbg,
        })
    append_log(LOG_PATH, log_buf)

    # Pass 3: SQL generation in batches
    pending = list(zip(qlist_remaining, join_plans))
    for start in range(0, len(pending), GEN_BATCH):
        chunk = pending[start:start + GEN_BATCH]
        items = [(db_id, q["question"], q.get("evidence", "") or "", jp)
                 for ((_i, q), jp) in chunk]
        sqls = generate_sql_batch(items)
        for ((orig_i, q), _jp), sql in zip(chunk, sqls):
            qid = str(q.get("question_id", orig_i))
            sql = (sql or "SELECT 1").replace("\n", " ").strip()
            predictions[qid] = f"{sql}\t----- bird -----\t{db_id}"
            pbar.update(1)
        gc.collect()
        try: torch.cuda.empty_cache()
        except Exception: pass
        if len(predictions) - last_checkpoint >= 50:
            write_preds(PRED_PATH, predictions)
            last_checkpoint = len(predictions)

    write_preds(PRED_PATH, predictions)
    print(f"  [{db_id}] done. {_mem_str()}")

pbar.close()
write_preds(PRED_PATH, predictions)
print(f"\nWrote {PRED_PATH} ({len(predictions)} predictions)")
print(f"Wrote {LOG_PATH} (graph debug)")
print(f"Memory at end: {_mem_str()}")

## 13. Local metrics — Join Acc + Exec Acc + Retrieval Recall + Graph Stats

Same EX/Join-F1 metrics as `retrieval.ipynb`, plus two graph-specific signals:

- **Graph-table recall:** did the graph-resolved set (anchors + bridges) include every gold table? Should be ≥ retrieval recall because of bridge insertion.
- **Mean graph-set size:** how many tables does the model see per question on average? Lower = more aggressive structural pruning.

In [ ]:
import sqlite3, json, time
import sqlglot
from sqlglot import exp as _exp

PRED_PATH = pathlib.Path("/content/predict_dev_graph.json")
LOG_PATH  = pathlib.Path("/content/graph_log.jsonl")
predictions = json.loads(PRED_PATH.read_text())
qmap = {str(q.get("question_id", i)): q for i, q in enumerate(questions)}

graph_log = {}
if LOG_PATH.exists():
    for line in LOG_PATH.read_text().splitlines():
        if line.strip():
            try:
                obj = json.loads(line); graph_log[obj["qid"]] = obj
            except Exception:
                pass

TIMEOUT_S = 10
SQLITE_PROGRESS_STEPS = 10_000
_NULL = "\x00NULL\x00"

def run_sql(db_id, sql):
    db_path = BIRD_DBS / db_id / f"{db_id}.sqlite"
    conn = sqlite3.connect(str(db_path), timeout=TIMEOUT_S)
    try:
        conn.execute(f"PRAGMA busy_timeout = {TIMEOUT_S * 1000}")
        deadline = time.monotonic() + TIMEOUT_S
        conn.set_progress_handler(lambda: 1 if time.monotonic() > deadline else 0, SQLITE_PROGRESS_STEPS)
        return conn.execute(sql).fetchall(), None
    except Exception as exc:
        return None, str(exc)
    finally:
        try: conn.set_progress_handler(None, 0)
        except Exception: pass
        conn.close()

def normalize(rows):
    if rows is None:
        return None
    return sorted(tuple(_NULL if c is None else str(c) for c in r) for r in rows)

def extract_joins(sql):
    try:
        tree = sqlglot.parse_one(sql, dialect="sqlite")
    except Exception:
        return set()
    alias_map = {}
    for t in tree.find_all(_exp.Table):
        n = (t.name or "").lower(); a = (t.alias or t.name or "").lower()
        if n and a:
            alias_map[a] = n
    joins = set()
    for eq in tree.find_all(_exp.EQ):
        l, r = eq.this, eq.expression
        if isinstance(l, _exp.Column) and isinstance(r, _exp.Column):
            lt = (l.table or "").lower(); rt = (r.table or "").lower()
            if lt and rt and lt in alias_map and rt in alias_map and lt != rt:
                joins.add(frozenset([f"{alias_map[lt]}.{l.name.lower()}",
                                     f"{alias_map[rt]}.{r.name.lower()}"]))
    return joins

def join_f1(pred_sql, gold_sql):
    gold = extract_joins(gold_sql)
    if not gold:
        return None
    pred = extract_joins(pred_sql)
    if not pred:
        return 0.0
    tp = len(pred & gold)
    if tp == 0:
        return 0.0
    p = tp / len(pred); r = tp / len(gold)
    return 2 * p * r / (p + r)

def gold_tables(sql):
    try:
        tree = sqlglot.parse_one(sql, dialect="sqlite")
    except Exception:
        return set()
    return {(t.name or "").lower() for t in tree.find_all(_exp.Table) if t.name}

results = []
for i, q in enumerate(questions):
    qid = str(q.get("question_id", i))
    line = predictions.get(qid)
    if line is None:
        pred_sql = "SELECT 1"; missing = True
    else:
        pred_sql = line.split("\t----- bird -----\t", 1)[0].strip(); missing = False
    is_fallback = pred_sql.rstrip(";").strip().upper() == "SELECT 1"

    pred_rows, pred_err = run_sql(q["db_id"], pred_sql)
    gold_rows, gold_err = run_sql(q["db_id"], q["SQL"])

    if missing: ex = "missing_prediction"
    elif pred_err:      ex = "exec_error"
    elif gold_err:      ex = "gold_error"
    elif normalize(pred_rows) == normalize(gold_rows): ex = "correct"
    else:               ex = "wrong_result"

    g = graph_log.get(qid, {})
    retrieved_set = {t.lower() for t in g.get("retrieved", [])}
    graph_set     = {t.lower() for t in g.get("graph_tables", [])}
    gold_set      = gold_tables(q["SQL"])
    ret_recall   = (len(retrieved_set & gold_set) / len(gold_set)) if gold_set else None
    graph_recall = (len(graph_set & gold_set)     / len(gold_set)) if gold_set else None

    results.append({
        "qid": qid, "db_id": q["db_id"], "difficulty": q.get("difficulty", "?"),
        "ex": ex, "pred_err": pred_err, "fallback": is_fallback, "missing": missing,
        "jf1": join_f1(pred_sql, q["SQL"]), "pred_sql": pred_sql,
        "ret_recall": ret_recall, "graph_recall": graph_recall,
        "n_retrieved": len(retrieved_set), "n_graph": len(graph_set),
        "n_bridges": len(g.get("bridges", [])),
        "graph_tables": list(graph_set), "gold_tables": list(gold_set),
        "log": g,
    })

total = len(results)
ex_correct = sum(r["ex"] == "correct" for r in results)
ex_acc = ex_correct / max(total, 1) * 100
scored_join = [r for r in results if r["jf1"] is not None]
join_acc = (sum(r["jf1"] for r in scored_join) / max(len(scored_join), 1)) * 100
scored_rr = [r for r in results if r["ret_recall"] is not None]
ret_recall_avg   = (sum(r["ret_recall"] for r in scored_rr) / max(len(scored_rr), 1)) * 100
graph_recall_avg = (sum(r["graph_recall"] for r in scored_rr) / max(len(scored_rr), 1)) * 100
mean_n_graph = sum(r["n_graph"] for r in results) / max(total, 1)
mean_n_bridges = sum(r["n_bridges"] for r in results) / max(total, 1)

print(f"=== Graph ablation (Qwen-32B + Qwen-7B exp + BM25/dense + Graph N=8) — {total} predictions ===\n")
print(f"{'Method':<32s}{'Join Acc.':>12s}{'Exec. Acc.':>13s}")
print(f"{'-'*57}")
print(f"{'Baseline (Qwen-32B, full)':<32s}{66.3:>12.1f}{54.2:>13.1f}")
print(f"{'Retrieval (N=5)':<32s}{66.7:>12.1f}{52.1:>13.1f}")
print(f"{'+ Graph (this run, N=8)':<32s}{join_acc:>12.1f}{ex_acc:>13.1f}")

print(f"\nUpstream signal:")
print(f"  Retrieval-table recall (top-8): {ret_recall_avg:.1f}%")
print(f"  Graph-table recall (anchors+bridges): {graph_recall_avg:.1f}%   (Δ from retrieval = +{graph_recall_avg - ret_recall_avg:.1f}pt from bridges)")
print(f"  Mean graph-set size: {mean_n_graph:.1f} tables  (mean bridges added: {mean_n_bridges:.2f})")
print(f"  Join Acc. averaged over {len(scored_join)} multi-table questions; {total - len(scored_join)} single-table excluded.")

print(f"\nEX status:  correct={ex_correct}  "
      f"wrong={sum(r['ex']=='wrong_result' for r in results)}  "
      f"err={sum(r['ex']=='exec_error' for r in results)}  "
      f"missing={sum(r['ex']=='missing_prediction' for r in results)}  "
      f"fallback={sum(r['fallback'] for r in results)}")

print(f"\n--- By difficulty ---")
print(f"{'difficulty':<14s}{'Join Acc.':>11s}{'Exec. Acc.':>12s}{'GraphRecall':>13s}{'cases':>8s}")
for d in ("simple", "moderate", "challenging", "?"):
    bucket = [r for r in results if r["difficulty"] == d]
    if not bucket:
        continue
    bj = [r for r in bucket if r["jf1"] is not None]
    j = (sum(r["jf1"] for r in bj) / max(len(bj), 1)) * 100 if bj else float("nan")
    e = sum(r["ex"] == "correct" for r in bucket) / len(bucket) * 100
    bg = [r for r in bucket if r["graph_recall"] is not None]
    gr = (sum(r["graph_recall"] for r in bg) / max(len(bg), 1)) * 100 if bg else float("nan")
    print(f"  {d:<12s}{j:>11.1f}{e:>12.1f}{gr:>13.1f}{len(bucket):>8d}")

print(f"\n--- By db_id ---")
print(f"{'db_id':<26s}{'Join Acc.':>11s}{'Exec. Acc.':>12s}{'GraphRecall':>13s}{'cases':>8s}")
by_db = defaultdict(list)
for r in results:
    by_db[r["db_id"]].append(r)
for d, bucket in sorted(by_db.items()):
    bj = [r for r in bucket if r["jf1"] is not None]
    j = (sum(r["jf1"] for r in bj) / max(len(bj), 1)) * 100 if bj else float("nan")
    e = sum(r["ex"] == "correct" for r in bucket) / len(bucket) * 100
    bg = [r for r in bucket if r["graph_recall"] is not None]
    gr = (sum(r["graph_recall"] for r in bg) / max(len(bg), 1)) * 100 if bg else float("nan")
    print(f"  {d:<24s}{j:>11.1f}{e:>12.1f}{gr:>13.1f}{len(bucket):>8d}")

print("\n--- First 5 EX failures (graph diagnosis) ---")
shown = 0
for r in results:
    if r["ex"] == "correct" or shown >= 5:
        continue
    q = qmap[r["qid"]]
    jf = "—" if r["jf1"] is None else f"{r['jf1']:.2f}"
    rr = "—" if r["ret_recall"] is None else f"{r['ret_recall']*100:.0f}%"
    gr = "—" if r["graph_recall"] is None else f"{r['graph_recall']*100:.0f}%"
    missing = set(r["gold_tables"]) - set(r["graph_tables"])
    g = r["log"]
    print(f"\n  qid={r['qid']} ({r['difficulty']}) db={r['db_id']}  ex={r['ex']}  jf1={jf}  ret={rr} graph={gr}")
    print(f"    Q:    {q['question'][:140]}")
    if q.get("evidence"):
        print(f"    ev:   {q['evidence'][:140]}")
    print(f"    retr: {', '.join(g.get('retrieved', [])) or '(none)'}")
    print(f"    graph:{', '.join(g.get('graph_tables', [])) or '(none)'}  (+{len(g.get('bridges', []))} bridges)")
    print(f"    gold: {', '.join(sorted(r['gold_tables'])) or '(none)'}")
    if missing:
        print(f"    *** GRAPH MISSED: {', '.join(sorted(missing))}")
    print(f"    joins:{'; '.join(g.get('approved_joins', [])) or '(no joins required)'}")
    print(f"    pred: {r['pred_sql'][:160]}")
    print(f"    gold: {q['SQL'][:160]}")
    if r["pred_err"]:
        print(f"    err:  {r['pred_err'][:160]}")
    shown += 1

## 14. Run BIRD's official evaluation scripts

Drop `evaluation.py` and `evaluation_ves.py` from BIRD's `evaluation/` folder into `/content/bird_eval/`, then run.

In [ ]:
BIRD_EVAL = pathlib.Path("/content/bird_eval")
BIRD_EVAL.mkdir(exist_ok=True)

GT_PATH = BIRD_EVAL / "dev_gold.sql"
lines = []
for q in questions:
    sql = q["SQL"].replace("\n", " ").strip()
    lines.append(f"{sql}\t{q['db_id']}")
GT_PATH.write_text("\n".join(lines))
print("Wrote", GT_PATH)

ex_script  = BIRD_EVAL / "evaluation.py"
ves_script = BIRD_EVAL / "evaluation_ves.py"

if ex_script.exists():
    !python {ex_script} \
        --predicted_sql_path {PRED_PATH} \
        --ground_truth_path {GT_PATH} \
        --db_root_path {BIRD_DBS} \
        --num_cpus 4 \
        --meta_time_out 30.0 \
        --mode_gt gt --mode_predict gpt
else:
    print(f"Place evaluation.py at {ex_script} and re-run.")

if ves_script.exists():
    !python {ves_script} \
        --predicted_sql_path {PRED_PATH} \
        --ground_truth_path {GT_PATH} \
        --db_root_path {BIRD_DBS} \
        --num_cpus 4 \
        --meta_time_out 30.0 \
        --mode_gt gt --mode_predict gpt
else:
    print(f"Place evaluation_ves.py at {ves_script} and re-run.")

## Notes

**Component contribution this ablation tests:** Δ EX between this run and the retrieval-only run quantifies what the graph step adds. The graph contributes:
- *Approved-joins constraint* (model can't fabricate join predicates)
- *Bridge-table insertion* (recovers some Source 1 recall losses via FK traversal)
- *Flat ordering* (eliminates Source 3 rank-demotion)

**Why N=8 retrieval (not 5):** at N=5 the graph would inherit retrieval's recall failures with no chance to recover. At N=8 retrieval recall is ~97-99% on every BIRD dev DB; the graph can prune the over-inclusion structurally. Bumping to N=12 nullifies the graph's pruning role on most DBs.

**Knobs:**
- `N_RETRIEVAL_TABLES` (cell 18) — currently 8. Trade-off: larger = more recall, less graph pruning room.
- `GRAPH_GEN_BATCH`, `GRAPH_EXP_BATCH` env vars — batch sizes.
- The graph implementation comes from `text2sql_agent_prototype/prototype.py:SchemaGraph`; modifying its `minimal_connecting_subgraph` strategy (e.g., to bound bridge-table count) would change behavior here.